In [1]:
import speech_recognition as srec
from gtts import gTTS
import pyttsx3 as pyt

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, VitsModel
import torch

import time
import psutil

import whisper
import sounddevice as sd
import numpy as np
from IPython.display import Audio
from openai import OpenAI

c:\Users\Alysha\Documents\kata-ondevice\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
engine = pyt.init()
voices = engine.getProperty('voices')
engine.setProperty('voice', voices[1].id)

device = "auto"

In [2]:
# --- LLAMA ---

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", torch_dtype=torch.bfloat16, 
  device_map='cpu')

In [4]:
client = OpenAI(
    base_url="http://localhost:8080/v1", # "http://<Your api-server IP>:port"
    api_key = "sk-no-key-required"
)

In [5]:
# --- STT ---

whisper_model = whisper.load_model("small")

def perintah():
    duration = 5
    sample_rate = 16000  

    print("Mendengarkan......")
    audio_data = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()  
    print("Diterima.....")

    audio_data = np.squeeze(audio_data)  
    dengar = whisper_model.transcribe(audio_data, fp16=False, language="id")
    print(dengar["text"])

    return dengar["text"]

c:\Users\Alysha\Documents\kata-ondevice\myenv\lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(fp, map_locati

In [ ]:
# # --- TTS ---

# def ngomong(text):
#     voices = engine.getProperty('voices')

#     for voice in voices:
#         if "MSTTS_V110_idID_Andika" in voice.id:
#             engine.setProperty('voice', voice.id)
#             break

#     # Speak the text
#     engine.say(text)
    
#     # Wait until speaking is finished
#     engine.runAndWait()

# # mms = VitsModel.from_pretrained("facebook/mms-tts-ind")
# # mms_token = AutoTokenizer.from_pretrained("facebook/mms-tts-ind")

# # def ngomong(text):
# #     inputs = mms_token(text, return_tensors="pt")

# #     with torch.no_grad():
# #         output = mms(**inputs).waveform
    
# #     return Audio(output.squeeze().cpu().numpy(), rate=16000) # Can change rate to make it faster/slower

In [6]:
def ngomong(text):
    # Ensure the text is a plain string
    if isinstance(text, dict) and "content" in text:
        text = text["content"]
    elif hasattr(text, "content"):
        text = text.content  # Extract content from ChatCompletionMessage object

    voices = engine.getProperty('voices')
    for voice in voices:
        if "MSTTS_V110_idID_Andika" in voice.id:
            engine.setProperty('voice', voice.id)
            break

    # Speak the plain text
    engine.say(text)
    engine.runAndWait()

In [13]:
def generate_response(user_input):
    start_time = time.time()
    memory_before = psutil.virtual_memory().used

    response = client.chat.completions.create(
        model="Llama-3.2-1B-Instruct-llamafile",
        messages=[
            {"role": "system", "content": "Talk like you are a voice assistant."},
            {"role": "user", "content": user_input},
        ],
    )

    end_time = time.time()
    memory_after = psutil.virtual_memory().used

    inference_time = end_time - start_time
    memory_used = memory_after - memory_before

    metrics = {
        "Inference Time": f"{inference_time:.2f} seconds",
        "Memory Usage": f"{memory_used / (1024 ** 2):.2f} MB",
    }

    speak = response.choices[0].message.content.replace("<|eot_id|>", "").strip()
    print("Response:", speak)
    print("Performance Metrics:", metrics)

    return speak, metrics


In [8]:
# --- Main Function ---
def run_va():
    # Get user input via STT
    user_input = perintah()

    # Generate response using Llamafile API
    response, metrics = generate_response(user_input)

    # Print metrics
    print("INFERENCE TIME:", metrics["Inference Time"])
    print("MEMORY USAGE:", metrics["Memory Usage"])

    # Speak the response
    ngomong(response)

In [ ]:
# # --- MAIN FUNCTION ---

# def run_va():
#     # prompt from user
#     Layanan = perintah()

#     messages = [
#     {"role": "system", "content": "Tolong jawab singkat."},
#     {"role": "user", "content": Layanan}
# ]

#     text = tokenizer.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True
#     )

#     model_inputs = tokenizer([text], return_tensors="pt") 

#     start_time = time.time()
#     memory_before = psutil.virtual_memory().used

#     generated_ids = model.generate(
#         model_inputs.input_ids,
#         max_new_tokens=512, 
#         temperature=0.7,
#         top_p=0.9,
#         do_sample=True

#     )

#     generated_ids = [
#         output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
#     ]
#     response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

#     end_time = time.time()
#     memory_after = psutil.virtual_memory().used

#     inference_time = end_time - start_time
#     memory_used = memory_after - memory_before
#     cpu_usage = psutil.cpu_percent(interval=1)
    
#     print(response)
#     print("INFERENCE TIME: ", inference_time)
#     print("MEMORY USAGE: ", memory_used)
#     print("CPU USAGE: ", cpu_usage)
#     ngomong(response)

In [14]:
run_va()

Mendengarkan......
Diterima.....
 bagaimana cara masak masih goreng
Response: Baik, saya akan menjelaskan cara masak still goreng yang lezat!

Still goreng adalah metode masak yang menggunakan api sedang untuk memasak makanan, sehingga makanan tetap tercium kecoklatan dan garing. Berikut adalah beberapa tips untuk membuat still goreng yang lezat:

1. **Pilih bahan yang tepat**: Pilih bahan yang memiliki tekstur yang baik dan dapat diolah dengan baik, seperti daging, sayuran, dan bumbu-bumbu.
2. **Gunakan minyak yang tepat**: Gunakan minyak yang memiliki kandungan lemak yang tinggi, seperti minyak zaitun atau minyak kelapa, untuk membuat makanan tercium kecoklatan.
3. **Tambahkan bumbu-bumbu yang tepat**: Tambahkan bumbu-bumbu yang memiliki rasa yang kuat, seperti gula, garam, dan merica, untuk membuat makanan tercium rasa yang lezat.
4. **Masak dengan api sedang**: Masak dengan api sedang, sehingga makanan tidak terlalu matang atau terlalu kering.
5. **Tutup api jika perlu**: Jika perl

KeyboardInterrupt: 